# ProductSearch — Hybrid Product Search Engine
Combines **fuzzy matching**, **keyword expansion**, and **semantic similarity** to search SBI YONO Business products.

---

## 1. Imports

In [1]:
import re
import os
import sys
import nltk
import sqlite3
import oracledb
import pandas as pd
import ipywidgets as widgets

from thefuzz import fuzz
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from itertools import combinations
from IPython.display import display
from collections import defaultdict
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from sentence_transformers import SentenceTransformer, util

## 2. Configuration & Mappings

In [2]:
# ──────────────────────────────────────────────
# 2a. Search Thresholds
# ──────────────────────────────────────────────
class SearchConfig:
    """Central configuration for all search thresholds."""

    FUZZY_THRESHOLDS = {
        'short':  85,   # word length <= 4
        'medium': 81,   # word length <= 6
        'long':   81,   # word length >  6
    }
    WORD_MATCH_THRESHOLD  = 85
    FUZZY_RATIO_THRESHOLD = 81
    SYNONYM_THRESHOLD     = 81
    SEMANTIC_THRESHOLD    = 0.6
    DEFAULT_THRESHOLD     = 81

    LENGTH_SHORT  = 4
    LENGTH_MEDIUM = 6

    @classmethod
    def get_fuzzy_threshold(cls, word_length: int) -> int:
        if word_length <= cls.LENGTH_SHORT:
            return cls.FUZZY_THRESHOLDS['short']
        elif word_length <= cls.LENGTH_MEDIUM:
            return cls.FUZZY_THRESHOLDS['medium']
        return cls.FUZZY_THRESHOLDS['long']

In [3]:
lemmatizer = WordNetLemmatizer()
# ──────────────────────────────────────────────
# 2b. Abbreviation Map  (abbr → full phrase)
# ──────────────────────────────────────────────
ABBREVIATION_MAP = {
    "ca":    "current account",
    "yb":    "yono business",
    "cmp":   "cash management product",
    "scf":   "supply chain finance",
    "lc":    "letter of credit",
    "fbg":   "foreign bank guarantee",
    "ebg":   "electronic bank guarantee",
    "ftc":   "foreign travel card",
    "ncd":   "Non-Convertible Debentures",
    "pabl":  "pre-approved business loan",
    "dde":   "digital document execution",
    "van":   "virtual account number",
    "ccpap": "corporate cheques payable at par",
    "edfs":  "electronic dealer financing scheme",
    "evfs":  "electronic vendor financing scheme",
    "whr":   "warehouse receipts",
    "treds": "trade receivable discounting system",
    "lap":   "loan against property"
}

# full phrase → abbreviation (auto-built, all lowercase)
REVERSE_ABBR_MAP = {v.lower(): k for k, v in ABBREVIATION_MAP.items()}

SYNONYM_MAP = {
    "loan": ["financing", "funding", "asset", "debt", "credit", "lending"],
    "whr": ["commodity"],
    "evfs": ["reverse factoring", "invoicemart registration"],
    "account": ["current account", "ca"],
    "merchant": ["trader", "wholesaler", "dealer", "retailer"],
    "payment": ["payout", "transfer", "salary"],
    }
REVERSE_SYNONYM_MAP = defaultdict(list)
for main_word, synonyms in SYNONYM_MAP.items():
    for syn in synonyms:
        raw = syn.lower().strip()
        if main_word not in REVERSE_SYNONYM_MAP[raw]:
            REVERSE_SYNONYM_MAP[raw].append(main_word)
        lemmatized = lemmatizer.lemmatize(raw)
        if lemmatized != raw:
            if main_word not in REVERSE_SYNONYM_MAP[lemmatized]:
                REVERSE_SYNONYM_MAP[lemmatized].append(main_word)

## 3. Data Loading

In [4]:
# Connect to local database
conn = sqlite3.connect('products.db')

# Load into DataFrame — rest of the notebook stays the same
df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

# Same cleanup as before
#df = df.reset_index(drop=True)
#df.columns = df.columns.str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)
#df = df.replace('Â', '', regex=True)

print(f"Product catalogue loaded — {len(df)} rows, {len(df.columns)} columns")
print(f"   Columns: {list(df.columns[:3])}\n")

Product catalogue loaded — 91 rows, 3 columns
   Columns: ['Product Name', 'Product Description', 'Display URL']



In [5]:
# # ──────────────────────────────────────────────
# # 3b. Oracle Connection — fill these variables
# # ──────────────────────────────────────────────
# DB_USERNAME = your_username
# DB_PASSWORD = your password
# DB_URL = your_connection_string

# # Connect to Oracle
# conn = oracledb.connect(
#     user     = DB_USERNAME,
#     password = DB_PASSWORD,
#     url      = DB_URL
# )

# # Load into DataFrame — rest of notebook stays the same
# df = pd.read_sql(
#     "SELECT PRODUCT_NAME, PRODUCT_DESC, URL, KEYWORD FROM SEARCH_MASTER",
#     conn
# )
# conn.close()

# # Rename columns to match rest of your notebook
# df = df.rename(columns={
#     "PRODUCT_NAME" : "Product Name",
#     "PRODUCT_DESC" : "Product Description",
#     "URL"          : "Display URL",
#     "KEYWORD"      : "Keyword"
# })

# df = df.reset_index(drop=True)
# df.columns = df.columns.str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)
# df = df.replace('Â', '', regex=True)

# print(f" Product catalogue loaded — {len(df)} rows, {len(df.columns)} columns")
# print(f"   Columns: {list(df.columns[:3])}\n")

## 4. Text Normalisation

In [6]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))
 
KEEP_POS_TAGS = {
    'NN', 'NNS', 'NNP', 'NNPS',
    'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ',
    'JJ', 'JJR', 'JJS',
    'RB', 'RBR', 'RBS',
    'CD',
}
 
 
def get_wordnet_pos(treebank_tag: str) -> str:
    """Map Penn Treebank POS tags → WordNet POS tags (required by lemmatizer)."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
 
 
def normalize_word(word: str, pos: str = wordnet.NOUN) -> str:
    """Lowercase and lemmatize a single token."""
    clean = word.lower().strip()
    if not clean or clean in stop_words:
        return ""
    return lemmatizer.lemmatize(clean, pos)
 
 
def normalize_text(text: str) -> str:
    """
    FIX: Correct order — tokenize → POS tag (on full text) → filter POS
         → lemmatize → THEN remove stopwords.
    Previously stopwords were removed BEFORE POS tagging, which broke
    grammatical context and caused mislabeling.
    """
    if not str(text).strip():
        return ""
 
    # STEP 1: Tokenize on the FULL text (stopwords still present)
    tokens = word_tokenize(str(text).lower())
 
    # STEP 2: POS tag on the FULL token list (correct grammatical context)
    pos_tags = pos_tag(tokens)
 
    # STEP 3: Filter by desired POS tags
    filtered = [(word, pos) for word, pos in pos_tags if pos in KEEP_POS_TAGS]
 
    # STEP 4: Lemmatize using correct POS, THEN remove stopwords
    result = []
    for word, pos in filtered:
        if word in stop_words:
            continue
        lemma = lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        if lemma and lemma not in stop_words:
            result.append(lemma)
 
    return " ".join(result)
 
 
def normalize_map_keys(mapping: dict) -> dict:
    """Normalize all keys in a dictionary."""
    return {normalize_text(k): v for k, v in mapping.items()}
 
 
NORM_ABBR_MAP = {k: normalize_text(v) for k, v in ABBREVIATION_MAP.items()}
NORM_SYNONYM_MAP = {
    normalize_text(k): [normalize_text(s) for s in v]
    for k, v in SYNONYM_MAP.items()
}
 
# Reverse: normalized synonym → list of normalized root words
NORM_REVERSE_SYNONYM_MAP: dict[str, list[str]] = {}
for root, synonyms in SYNONYM_MAP.items():
    for syn in synonyms:
        if not syn:
            continue
        for key in {syn.lower().strip(), normalize_text(syn)}:  # set deduplicates identical keys
            if key:
                NORM_REVERSE_SYNONYM_MAP.setdefault(key, [])
                if root not in NORM_REVERSE_SYNONYM_MAP[key]:
                    NORM_REVERSE_SYNONYM_MAP[key].append(root)
 
df['Normalized_Name'] = df['Product Name'].apply(normalize_text)
df['Normalized_Desc'] = df['Product Description'].apply(normalize_text)
#print("Text normalization complete.")

## 5. Semantic Model

In [ ]:
# Download: pip install sentence-transformers
# model = SentenceTransformer("BAAI/bge-base-en-v1.5")  # downloads automatically
model_path = r"path/to/your/local/bge-base-en-v1.5"     # or use the line above
model = SentenceTransformer(model_path)

In [8]:
#df['combined_text'] = (
#    df['Product Name'].fillna('') + " " +
#    df['Product Description'].fillna('')
#)

corpus_embeddings = model.encode(
    #df['combined_text'].tolist(),
    #df['Product Description'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=False
)

#print(f"Corpus embeddings ready — shape: {corpus_embeddings.shape}")

## 6. Search Engine

In [9]:
# import json

# data = {"queries": ["supply chain finance"]}

# with open("input.json", "w") as f:
#     json.dump(data, f)

In [10]:
def expand_keywords(keywords: list[str]) -> list[str]:
    """Expand keywords by looking up abbreviations and synonyms."""
    expanded: set[str] = set()
 
    def _add_with_mappings(term: str):
        term_l = term.lower().strip()
        if not term_l:
            return
        expanded.add(term_l)
        if term_l in ABBREVIATION_MAP:
            expanded.add(ABBREVIATION_MAP[term_l].lower())
        if term_l in REVERSE_ABBR_MAP:
            expanded.add(REVERSE_ABBR_MAP[term_l])
        if term_l in SYNONYM_MAP:
            for syn in SYNONYM_MAP[term_l]:
                expanded.add(syn.lower())
        if term_l in REVERSE_SYNONYM_MAP:
            for main_word in REVERSE_SYNONYM_MAP[term_l]:
                expanded.add(main_word.lower())
                if main_word.lower() in SYNONYM_MAP:
                    for sibling in SYNONYM_MAP[main_word.lower()]:
                        expanded.add(sibling.lower())
 
    def _add_normalized_with_mappings(term: str):
        norm = normalize_text(term)
        if not norm:
            return
        expanded.add(norm)
        if norm in NORM_ABBR_MAP:
            expanded.add(NORM_ABBR_MAP[norm])
        if norm in NORM_SYNONYM_MAP:
            for syn in NORM_SYNONYM_MAP[norm]:
                expanded.add(syn)
        if norm in NORM_REVERSE_SYNONYM_MAP:
            for main_word in NORM_REVERSE_SYNONYM_MAP[norm]:
                expanded.add(main_word)
                if main_word in NORM_SYNONYM_MAP:
                    for sibling in NORM_SYNONYM_MAP[main_word]:
                        expanded.add(sibling)
 
    def _fuzzy_abbreviation_expansion(term_l: str):
        abbr_threshold = SearchConfig.get_fuzzy_threshold(len(term_l))
        for abbr, full in ABBREVIATION_MAP.items():
            if fuzz.ratio(term_l, abbr) >= abbr_threshold:
                expanded.add(abbr.lower())
                expanded.add(full.lower())
        for full, abbr in REVERSE_ABBR_MAP.items():
            if fuzz.ratio(term_l, full) >= abbr_threshold:
                expanded.add(abbr.lower())
                expanded.add(full.lower())
                
    for kw in keywords:
        raw = kw.lower().strip()
        if not raw:
            continue
        _add_with_mappings(raw)
        _add_normalized_with_mappings(raw)
        _fuzzy_abbreviation_expansion(raw)
        tokens = raw.split()
        if len(tokens) > 1:
            expanded.add(raw)
            for tok in tokens:
                _add_with_mappings(tok)
                _add_normalized_with_mappings(tok)
                _fuzzy_abbreviation_expansion(tok)

    raw_joined = " ".join(k.lower().strip() for k in keywords)
    for full_phrase, abbr in REVERSE_ABBR_MAP.items():
        if full_phrase in raw_joined:
            expanded.add(full_phrase)
            expanded.add(abbr)

    return list(expanded)
 
 
def calculate_exact_match_score(query_words: list[str], target_text: str) -> tuple[float, list[str]]:
    target_tokens = target_text.split()
    matched_words = [w for w in query_words if w in target_tokens]
    score = (len(matched_words) / len(query_words)) * 100 if query_words else 0
    return score, matched_words
 
 
def calculate_fuzzy_match_score(
    query_words: list[str],
    target_text: str,
    threshold: int,
) -> tuple[float, list[str]]:
    target_tokens = target_text.split()
    matched_words = []
    best_scores   = []
 
    for query_word in query_words:
        best_score = max(
            (fuzz.token_set_ratio(query_word, token) for token in target_tokens),
            default=0,
        )
        if best_score >= threshold:
            matched_words.append(query_word)
            best_scores.append(best_score)
 
    if not query_words:
        return 0, []
 
    percentage_matched  = (len(matched_words) / len(query_words)) * 100
    average_fuzzy_score = sum(best_scores) / len(best_scores) if best_scores else 0
    combined_score      = (percentage_matched * average_fuzzy_score) / 100
    return combined_score, matched_words

ACTION_WORDS = {
    'check', 'apply', 'status', 'track', 'view', 'open',
    'verify', 'update', 'manage', 'download', 'upload',
    'resume', 'submit', 'register', 'request', 'initiate'
}
OPAQUE_SYNONYM_ROOTS = {}  # products users won't know by name

def calculate_intent_boost(query_keywords, product_name):
    product_name_lower = product_name.lower()
    boost = 0.0
    for kw in query_keywords:
        kw_lower = kw.lower()
        if kw_lower in REVERSE_SYNONYM_MAP:
            for root in REVERSE_SYNONYM_MAP[kw_lower]:
                if root.lower() in OPAQUE_SYNONYM_ROOTS and root.lower() in product_name_lower:
                    boost += 30.0
    return min(boost, 100.0)

def extract_key_phrases(
    query_words: list[str],
    min_phrase_len: int = 2,
    max_phrase_len: int = 3,
) -> list[str]:
    """Sliding window n-grams — extracts only real contiguous phrases."""
    normalized_words = [normalize_text(w) for w in query_words]
    normalized_words = [w for w in normalized_words if w.strip()]
 
    if len(normalized_words) < min_phrase_len:
        return []
 
    phrases = []
    for phrase_len in range(min_phrase_len, min(max_phrase_len + 1, len(normalized_words) + 1)):
        for start in range(len(normalized_words) - phrase_len + 1):
            phrase = " ".join(normalized_words[start : start + phrase_len])
            if phrase.strip():
                phrases.append(phrase)
 
    return list(dict.fromkeys(phrases))
 
 
def score_phrase_matches(
    phrases: list[str],
    target_text: str,
    threshold: int,
) -> dict[str, float]:
    target_lower  = target_text.lower()
    phrase_scores = {}
 
    for phrase in phrases:
        phrase_lower = phrase.lower()
        if phrase_lower in target_lower:
            phrase_scores[phrase] = 100.0
        else:
            phrase_tokens = phrase_lower.split()
            target_tokens = target_lower.split()
            best_score    = 0
            for i in range(len(target_tokens) - len(phrase_tokens) + 1):
                candidate  = " ".join(target_tokens[i : i + len(phrase_tokens)])
                best_score = max(best_score, fuzz.ratio(phrase_lower, candidate))
            if best_score >= threshold:
                phrase_scores[phrase] = best_score
 
    return phrase_scores
 
 
def _get_dynamic_threshold(word_len: int) -> int:
    return 85 if word_len <= 4 else 81
    
def hybrid_search(
    df: pd.DataFrame,
    query: str,
    corpus_embeddings,
    fuzzy_threshold: int   = 81,
    semantic_threshold: float = 0.6,
) -> pd.DataFrame:
 
    raw_query       = query.lower().strip()
    is_abbreviation = raw_query in ABBREVIATION_MAP or raw_query in REVERSE_ABBR_MAP
 
    # STEP 1: Keyword extraction and expansion
    if is_abbreviation:
        query_keywords = expand_keywords([raw_query])
    else:
        query_normalized = normalize_text(raw_query)
        initial_keywords = query_normalized.split()
        query_keywords   = expand_keywords(initial_keywords)
 
    # STEP 2: Phrase extraction — sliding window n-grams
    # Source A: original query word order → only real, meaningful phrases
    _original_tokens = normalize_text(raw_query).split()
    _original_phrases = extract_key_phrases(_original_tokens, min_phrase_len=2, max_phrase_len=3)
    
    # Source B: multi-word entries from synonym/abbr expansion ONLY
    _expansion_phrases = [
        exp for exp in query_keywords
        if " " in exp and exp not in raw_query.lower()]
    
    query_phrases = list(dict.fromkeys(_original_phrases + _expansion_phrases))
 
    # [Reasoning suppressed] Query debug info:
    # print(f"Query (raw)      : '{query}'")
    # print(f"Query (keywords) : {query_keywords}")
    # print(f"Query (phrases)  : {query_phrases}\n")
 
    # STEP 3: Semantic embeddings — encode raw + normalized query only (not expanded noise)
    normalized_query = normalize_text(raw_query)
    semantic_queries = list(dict.fromkeys([raw_query, normalized_query]))
    query_embeddings = model.encode(semantic_queries, convert_to_tensor=True)
    sem_matrix       = util.cos_sim(query_embeddings, corpus_embeddings)
 
    # STEP 4: Per-product scoring
    results_data = []
 
    for idx, row in df.iterrows():
        name_text = row["Normalized_Name"]
        desc_text = row["Normalized_Desc"]
 
        exact_name_score, exact_name_words = calculate_exact_match_score(query_keywords, name_text)
        exact_desc_score, exact_desc_words = calculate_exact_match_score(query_keywords, desc_text)
 
        max_word_len = max((len(w) for w in query_keywords), default=0)
        threshold    = _get_dynamic_threshold(max_word_len)

        intent_boost = calculate_intent_boost(query_keywords, row["Normalized_Name"])
 
        remaining_for_fuzzy_name = [w for w in query_keywords if w not in exact_name_words]
        remaining_for_fuzzy_desc = [w for w in query_keywords if w not in exact_desc_words]
 
        fuzzy_name_score, fuzzy_name_words = calculate_fuzzy_match_score(remaining_for_fuzzy_name, name_text, threshold)
        fuzzy_desc_score, fuzzy_desc_words = calculate_fuzzy_match_score(remaining_for_fuzzy_desc, desc_text, threshold)
 
        phrase_name_scores = score_phrase_matches(query_phrases, name_text, threshold)
        phrase_desc_scores = score_phrase_matches(query_phrases, desc_text, threshold)
 
        phrase_match_score = (
            (sum(phrase_name_scores.values()) + sum(phrase_desc_scores.values()))
            / (2 * len(query_phrases))
            if query_phrases else 0
        )
 
        semantic_scores           = [sem_matrix[q_idx][idx].item() for q_idx in range(len(semantic_queries))]
        best_semantic_score       = sum(semantic_scores) / len(semantic_scores)
        semantic_score_normalized = best_semantic_score * 100
        _name_word_count = max(len(normalize_text(row["Product Name"]).split()), 1)
        _name_matched    = len(exact_name_words) + len(fuzzy_name_words)
        name_density_bonus = (_name_matched / _name_word_count) * 30
        _first_word = normalize_text(row["Product Name"]).split()[0] if normalize_text(row["Product Name"]) else ""
        action_penalty = 15 if _first_word in ACTION_WORDS else 0
 
        # STEP 5: Combined score — phrase score included
        combined_score = (
            exact_name_score     *2
            + fuzzy_name_score   *1.3
            + exact_desc_score
            + fuzzy_desc_score
            + phrase_match_score
            + semantic_score_normalized
            + intent_boost
            + name_density_bonus
            - action_penalty 
        )
 
        # Out-of-scope guard (inner per-product gate)
        non_semantic_score = (
            exact_name_score + fuzzy_name_score +
            exact_desc_score + fuzzy_desc_score +
            phrase_match_score
        )
        total_match_count = (
            len(exact_name_words) + len(exact_desc_words) +
            len(fuzzy_name_words) + len(fuzzy_desc_words)
        )
 
        if non_semantic_score < 10 and total_match_count == 0:
            continue
 
        has_strong_keyword_match = non_semantic_score >= 10 or total_match_count > 0
        if not has_strong_keyword_match and best_semantic_score < SearchConfig.SEMANTIC_THRESHOLD:
            continue
 
        # STEP 6: Build match reason
        reason_parts = []
        if exact_name_score        > 0: reason_parts.append(f"Exact Name: {exact_name_score:.1f}%")
        if fuzzy_name_score        > 0: reason_parts.append(f"Fuzzy Name: {fuzzy_name_score:.1f}%")
        if exact_desc_score        > 0: reason_parts.append(f"Exact Desc: {exact_desc_score:.1f}%")
        if fuzzy_desc_score        > 0: reason_parts.append(f"Fuzzy Desc: {fuzzy_desc_score:.1f}%")
        if phrase_match_score      > 0: reason_parts.append(f"Phrase Match: {phrase_match_score:.1f}%")
        if semantic_score_normalized > 0: reason_parts.append(f"Semantic: {semantic_score_normalized:.1f}")
        if intent_boost > 0: reason_parts.append(f"Intent Boost: {intent_boost:.1f}")    
 
        match_reason = " | ".join(reason_parts) if reason_parts else "Match Found"
 
        results_data.append({
            "index"                    : idx,
            "combined_score"           : combined_score,
            "exact_name_score"         : exact_name_score,
            "fuzzy_name_score"         : fuzzy_name_score,
            "exact_desc_score"         : exact_desc_score,
            "fuzzy_desc_score"         : fuzzy_desc_score,
            "phrase_match_score"       : phrase_match_score,
            "semantic_score"           : best_semantic_score,
            "semantic_score_normalized": semantic_score_normalized,
            "match_count"              : (
                len(exact_name_words) + len(exact_desc_words)
                + len(fuzzy_name_words) + len(fuzzy_desc_words)
            ),
            "phrase_match_count"       : len(phrase_name_scores) + len(phrase_desc_scores),
            "match_reason"             : match_reason,
        })
 
    if not results_data:
        return pd.DataFrame()

    results_df_temp = pd.DataFrame(results_data)
    meaningful = results_df_temp[
        (results_df_temp['exact_name_score'] > 0) |
        (results_df_temp['exact_desc_score'] > 0) |
        (results_df_temp['semantic_score'] >= 0.55)
    ]
    if meaningful.empty:
        return pd.DataFrame()
    results_data = meaningful.to_dict('records')    
 
    # STEP 7: Sort
    temp = pd.DataFrame(results_data).sort_values(
        by=["combined_score", "phrase_match_score", "match_count"],
        ascending=[False, False, False],
    )
 
    result = df.loc[temp["index"].values].copy()
    result["Priority"]              = range(1, len(result) + 1)
    result["Combined Score"]        = temp["combined_score"].values.round(2)
    result["Exact Name Score"]      = temp["exact_name_score"].values.round(2)
    result["Fuzzy Name Score"]      = temp["fuzzy_name_score"].values.round(2)
    result["Exact Desc Score"]      = temp["exact_desc_score"].values.round(2)
    result["Fuzzy Desc Score"]      = temp["fuzzy_desc_score"].values.round(2)
    result["Phrase Match Score"]    = temp["phrase_match_score"].values.round(2)
    result["Semantic Score"]        = temp["semantic_score"].values.round(3)
    result["Semantic Score Normal"] = temp["semantic_score_normalized"].values.round(2)
    result["Match Reason"]          = temp["match_reason"].values
 
    return result
# ──────────────────────────────────────────────
# 7. Main: JSON Input → JSON Output
# ──────────────────────────────────────────────
def run_search(query: str, output_file: str = "search_results.json", top_n: int = 30):
    """
    Run a product search and write results to a JSON file.

    Args:
        query:       The user's search query string.
        output_file: Path for the output JSON file.
        top_n:       Maximum number of results to return.
    """
    results = hybrid_search(df, query, corpus_embeddings)

    if results.empty:
        output = {
            "query": query,
            "total_results": 0,
            "results": []
        }
    else:
        top_results = results.head(top_n)

        result_list = []
        for _, row in top_results.iterrows():
            result_list.append({
                "product_name": row.get("Product Name", ""),
                "url":          row.get("Display URL", ""),
            })

        output = result_list

    # Write to JSON output file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(json.dumps(output, indent=2, ensure_ascii=False))
    print(f"\n[Saved to {output_file}]", file=sys.stderr)

    return output


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage:")
        print('  Text input : python product_search.py "your search query"')
        print('  JSON input : python product_search.py input.json')
        sys.exit(1)

    arg = "input.json"          # ← change to your input file path, or a plain query string
    output_path = "search_results.json"
    
    if arg.endswith(".json") and os.path.isfile(arg):
        with open(arg, "r", encoding="utf-8") as f:
            input_data = json.load(f)
    
        if isinstance(input_data, list):
            queries = [item["query"] for item in input_data if "query" in item]
        elif "queries" in input_data:
            queries = input_data["queries"]
        elif "query" in input_data:
            queries = [input_data["query"]]
        else:
            raise ValueError("Input JSON must have a 'query' or 'queries' key.")
    
        all_results = []
        for q in queries:
            print(f"Searching: {q}")
            result = run_search(q, output_file="_tmp_result.json")
            all_results.append(result)
    
        merged = {
            "total_queries": len(all_results),
            "searches": all_results
        }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(merged, f, indent=2, ensure_ascii=False)
    
        if os.path.exists("_tmp_result.json"):
            os.remove("_tmp_result.json")
    
        #print(f"\nAll results saved to {output_path}")
    
    else:
        # plain text query
        run_search(arg, output_path) 

Searching: supply chain finance
[
  {
    "product_name": "Supply Chain Finance (SCF)",
    "url": "L3 Page of Supply Chain Finance (SCF)"
  },
  {
    "product_name": "Loan Against Warehouse Receipts (WHR Finance)",
    "url": "L3 Page of Loan Against Warehouse Receipts (WHR)"
  },
  {
    "product_name": "Electronic Vendor Financing Scheme (eVFS)",
    "url": "L3 Page of Electronic Vendor Financing Scheme (eVFS)"
  },
  {
    "product_name": "Trade Finance ( eTrade)",
    "url": "L3 Page of Trade Finance ( eTrade)"
  },
  {
    "product_name": "Electronic Dealer Financing Scheme (eDFS)",
    "url": "L3 Page of Electronic Dealer Financing Scheme (eDFS)"
  }
]



[Saved to _tmp_result.json]


In [12]:
# DISPLAY_COLUMNS = [
#     'Product Name',
#     'Product Description',
#     'Display URL',
#     'Priority',
#     'Exact Name Score',
#     'Fuzzy Name Score',
#     'Exact Desc Score',
#     'Fuzzy Desc Score',
#     'Semantic Score Normal',
#     'Combined Score',
#     'Match Reason'
# ]

# # ── Widgets ──
# search_box = widgets.Text(
#     placeholder='e.g. foreign travel insurance, lend, ca, scf …',
#     description='Search:',
#     layout=widgets.Layout(width='500px')
# )

# search_button = widgets.Button(
#     description='Search',
#     button_style='primary',
#     icon='search'
# )

# output = widgets.Output()
    
# # ── Callback ──
# def on_search_click(b):
#     with output:
#         output.clear_output()
#         query = search_box.value.strip()
#         if not query:
#             print("Please enter a search query.")
#             return
#         results = hybrid_search(df, query, corpus_embeddings)
#         if results.empty:
#             print(f'No results found for "{query}".')
#         else:
#             cols = [c for c in DISPLAY_COLUMNS if c in results.columns]
#             top_results = results.head(30)          # ← add this
#             print(f'Showing top {len(top_results)} of {len(results)} result(s) for "{query}"\n')
#             display(top_results[cols])             # ← change results to top_results

# search_button.on_click(on_search_click)

# # ── Render ──
# display(
#     widgets.HBox([search_box, search_button]),
#     output
# )